In [1]:
import pyodbc
from sqlalchemy import create_engine
import urllib
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Connect to our SQL Server instance
driver = "ODBC Driver 17 for SQL Server"
server_name = r"LAPTOP-5CHV2K6U\UMASANKAR"
database_name = "ShopSenseDB"

conn_str = (
    f"DRIVER={{{driver}}};"
    f"SERVER={server_name};"
    f"DATABASE={database_name};"
    f"Trusted_Connection=yes;"
    f"TrustServerCertificate=yes;"
)
engine = create_engine(f"mssql+pyodbc:///?odbc_connect={urllib.parse.quote_plus(conn_str)}")

# Pull transaction history for delivered orders
query = """
SELECT 
    customer_unique_id,
    order_id,
    order_purchase_timestamp,
    total_item_value
FROM dbo.vw_order_details
WHERE order_status = 'delivered';
"""

with engine.connect() as conn:
    df_sales = pd.read_sql(query, conn)

# Ensure datetime format
df_sales["order_purchase_timestamp"] = pd.to_datetime(df_sales["order_purchase_timestamp"])

print(f"✓ Pulled {len(df_sales):,} line items across {df_sales['customer_unique_id'].nunique():,} unique customers from SQL Server!")

✓ Pulled 110,197 line items across 93,358 unique customers from SQL Server!


In [2]:
# 1. Establish the anchor date (Max purchase date + 1 day)
snapshot_date = df_sales["order_purchase_timestamp"].max() + pd.Timedelta(days=1)
print(f"Analysis Snapshot Anchor Date: {snapshot_date.date()}")

# 2. Aggregate by customer_unique_id
rfm = df_sales.groupby("customer_unique_id").agg(
    recency=("order_purchase_timestamp", lambda dates: (snapshot_date - dates.max()).days),
    frequency=("order_id", "nunique"),
    monetary=("total_item_value", "sum")
).reset_index()

# Round monetary to 2 decimal places
rfm["monetary"] = rfm["monetary"].round(2)

print("--- RFM SUMMARY STATISTICS ---")
display(rfm[["recency", "frequency", "monetary"]].describe())

Analysis Snapshot Anchor Date: 2018-08-30
--- RFM SUMMARY STATISTICS ---


,recency,frequency,monetary
count,93358.000000,93358.000000,93358.000000
mean,237.941773,1.033420,165.168210
std,152.591453,0.209097,226.292101
min,1.000000,1.000000,9.590000
25%,114.000000,1.000000,63.010000
50%,219.000000,1.000000,107.780000
75%,346.000000,1.000000,182.510000
max,714.000000,15.000000,13664.080000


In [3]:
# 1. Recency Score (1 to 5): Lower days = Higher Score (more recent)
rfm["R_Score"] = pd.qcut(rfm["recency"], q=5, labels=[5, 4, 3, 2, 1]).astype(int)

# 2. Monetary Score (1 to 5): Higher spend = Higher Score
rfm["M_Score"] = pd.qcut(rfm["monetary"], q=5, labels=[1, 2, 3, 4, 5]).astype(int)

# 3. Frequency Score (Custom business logic for repeat orders)
def assign_f_score(freq):
    if freq >= 3:
        return 5
    elif freq == 2:
        return 3
    else:
        return 1

rfm["F_Score"] = rfm["frequency"].apply(assign_f_score)

# 4. Composite RFM Score String (e.g., '515', '111')
rfm["RFM_Group"] = rfm["R_Score"].astype(str) + rfm["F_Score"].astype(str) + rfm["M_Score"].astype(str)
rfm["RFM_Score"] = rfm[["R_Score", "F_Score", "M_Score"]].mean(axis=1).round(2)

# 5. Define Strategic Business Segments
def segment_customer(row):
    r = row["R_Score"]
    f = row["F_Score"]
    m = row["M_Score"]
    
    if r >= 4 and f >= 3 and m >= 4:
        return "Champions (VIP)"
    elif f >= 3:
        return "Loyal Customers"
    elif r >= 4 and f == 1:
        return "Promising New"
    elif r <= 2 and (f >= 3 or m >= 4):
        return "At Risk / High Value Lost"
    elif r <= 2 and f == 1 and m <= 2:
        return "Lost / Hibernating"
    else:
        return "Potential Loyalist"

rfm["Customer_Segment"] = rfm.apply(segment_customer, axis=1)

print("--- CUSTOMER SEGMENT BREAKDOWN ---")
segment_summary = rfm.groupby("Customer_Segment").agg(
    total_customers=("customer_unique_id", "count"),
    avg_recency_days=("recency", "mean"),
    avg_frequency=("frequency", "mean"),
    avg_monetary=("monetary", "mean"),
    total_revenue=("monetary", "sum")
).reset_index()

# Add percentage
segment_summary["customer_pct"] = (100.0 * segment_summary["total_customers"] / len(rfm)).round(2)
segment_summary["revenue_pct"] = (100.0 * segment_summary["total_revenue"] / rfm["monetary"].sum()).round(2)

segment_summary = segment_summary.sort_values(by="total_revenue", ascending=False)
display(segment_summary)

--- CUSTOMER SEGMENT BREAKDOWN ---


,Customer_Segment,total_customers,avg_recency_days,avg_frequency,avg_monetary,total_revenue,customer_pct,revenue_pct
5,Promising New,36224,90.621549,1.000000,164.329703,5952679.15,38.80,38.60
0,At Risk / High Value Lost,13735,394.169348,1.000000,307.884456,4228793.00,14.71,27.42
4,Potential Loyalist,25302,270.165481,1.000000,139.194628,3521902.49,27.10,22.84
2,Lost / Hibernating,15296,396.469992,1.000000,55.714674,852211.65,16.38,5.53
3,Loyal Customers,1811,292.162341,2.076201,273.807118,495864.69,1.94,3.22
1,Champions (VIP),990,88.827273,2.182828,372.043202,368322.77,1.06,2.39


In [4]:
# Save into SQL Server as a permanent table
print("Writing [dbo.customer_rfm_segments] into SQL Server...")

rfm.to_sql(
    name="customer_rfm_segments", 
    con=engine, 
    if_exists="replace", 
    index=False, 
    chunksize=10000
)

print(f"✓ Successfully stored {len(rfm):,} customer intelligence profiles into SQL Server!")

Writing [dbo.customer_rfm_segments] into SQL Server...
✓ Successfully stored 93,358 customer intelligence profiles into SQL Server!
